In [ ]:
# ============================================================
#  PHASE 3 — XGBOOST MODEL TRAINING & EVALUATION
#  Stock Price Prediction & Analysis System
# ============================================================
# Prerequisites: phase2_feature_engineering.py must be run first
# Input : data/processed/master_features.csv
# Output: models/xgb_large.pkl, xgb_mid.pkl, xgb_small.pkl
#         models/universal_xgb.pkl  reports/model_report.csv
# ============================================================

# ── CELL 1: Install ──────────────────────────────────────────
# !pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib shap tqdm ta

# ── CELL 2: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for Jupyter too)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
from tqdm import tqdm

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics         import (accuracy_score, classification_report,
                                      confusion_matrix, roc_auc_score,
                                      precision_score, recall_score, f1_score)
from sklearn.preprocessing   import RobustScaler
from sklearn.pipeline        import Pipeline
from xgboost                 import XGBClassifier
import shap

warnings.filterwarnings("ignore")
os.makedirs("models",  exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("plots",   exist_ok=True)

# ── CELL 3: Columns ──────────────────────────────────────────
FEATURE_COLS = [
    "EMA50","EMA200","MACD","MACD_Signal","EMA_RATIO",
    "RSI14","Stoch_K","Stoch_D","ROC10","MOM10",
    "ATR14","BB_Upper","BB_Lower","BB_Width","RollingStd20",
    "OBV","VWAP","Vol_Ratio20",
    "Cap_Encoded",
]
TARGET = "Target"

# ── CELL 4: Smart Loader — builds features if missing ────────
def compute_indicators_single(df: pd.DataFrame) -> pd.DataFrame:
    """Fallback: compute all 18 indicators on a raw OHLCV dataframe."""
    from ta.trend      import EMAIndicator, MACD as MACDInd
    from ta.momentum   import RSIIndicator, StochasticOscillator
    from ta.volatility import AverageTrueRange, BollingerBands

    df = df.copy().sort_values("Date").reset_index(drop=True)
    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]
    volume = df["Volume"]

    # Trend
    df["EMA50"]       = EMAIndicator(close, 50,  fillna=True).ema_indicator()
    df["EMA200"]      = EMAIndicator(close, 200, fillna=True).ema_indicator()
    _m                = MACDInd(close, 26, 12, 9, fillna=True)
    df["MACD"]        = _m.macd()
    df["MACD_Signal"] = _m.macd_signal()
    df["EMA_RATIO"]   = df["EMA50"] / df["EMA200"]

    # Momentum
    df["RSI14"]   = RSIIndicator(close, 14, fillna=True).rsi()
    _s            = StochasticOscillator(high, low, close, 14, 3, fillna=True)
    df["Stoch_K"] = _s.stoch()
    df["Stoch_D"] = _s.stoch_signal()
    df["ROC10"]   = close.pct_change(10) * 100
    df["MOM10"]   = close.diff(10)

    # Volatility
    df["ATR14"]        = AverageTrueRange(high, low, close, 14, fillna=True).average_true_range()
    _bb                = BollingerBands(close, 20, 2, fillna=True)
    df["BB_Upper"]     = _bb.bollinger_hband()
    df["BB_Lower"]     = _bb.bollinger_lband()
    df["BB_Width"]     = _bb.bollinger_wband()
    df["RollingStd20"] = close.rolling(20).std()

    # Volume
    df["OBV"]         = (np.sign(close.diff()) * volume).fillna(0).cumsum()
    df["VWAP"]        = (close * volume).rolling(20).sum() / volume.rolling(20).sum()
    df["Vol_Ratio20"] = volume / volume.rolling(20).mean()

    # Target
    df["Daily_Return"] = close.pct_change() * 100
    df["Target"]       = (df["Daily_Return"].shift(-1) > 0.35).astype(int)

    return df


def load_master() -> pd.DataFrame:
    """
    Load master_features.csv if it exists (Phase 2 output).
    If not, fall back to master_raw.csv and compute indicators on-the-fly.
    """
    features_path = "data/processed/master_features.csv"
    raw_path      = "data/raw/master_raw.csv"

    # ── Case 1: features file exists ────────────────────────
    if os.path.exists(features_path):
        df = pd.read_csv(features_path, parse_dates=["Date"])
        print(f"✅  Loaded features file: {len(df):,} rows")

        # Verify required columns exist
        missing_cols = [c for c in FEATURE_COLS + [TARGET] if c not in df.columns]
        if not missing_cols:
            print(f"    All {len(FEATURE_COLS)} feature columns + Target present.")
            return df
        else:
            print(f"⚠️  Features file is missing columns: {missing_cols}")
            print("    Re-computing indicators from scratch...")

    # ── Case 2: only raw file exists — compute indicators ───
    if not os.path.exists(raw_path):
        raise FileNotFoundError(
            "Neither data/processed/master_features.csv nor data/raw/master_raw.csv found.\n"
            "Please run Phase 1 (phase1_data_ingestion.py) first."
        )

    print(f"⚠️  master_features.csv not found — loading raw data and computing indicators.")
    print(f"    (This means Phase 2 was not run. Computing indicators now...)\n")

    raw = pd.read_csv(raw_path, parse_dates=["Date"])

    # Basic cleaning: forward-fill, drop nulls
    raw.sort_values(["Ticker","Date"], inplace=True)
    raw[["Open","High","Low","Close"]] = (
        raw.groupby("Ticker")[["Open","High","Low","Close"]]
           .transform(lambda x: x.fillna(method="ffill").fillna(method="bfill"))
    )
    raw["Volume"] = raw["Volume"].fillna(0)
    raw = raw.dropna(subset=["Close"])
    raw = raw[raw["Close"] > 0]

    # Cap encoding
    cap_map = {"large": 0, "mid": 1, "small": 2}
    if "Cap_Type" not in raw.columns:
        raw["Cap_Type"]    = "large"
    raw["Cap_Encoded"] = raw["Cap_Type"].map(cap_map).fillna(0).astype(int)

    # Compute indicators per ticker
    print("Computing 18 indicators per ticker...")
    dfs = []
    for ticker, grp in tqdm(raw.groupby("Ticker"), desc="Indicators"):
        result = compute_indicators_single(grp)
        dfs.append(result)

    master = pd.concat(dfs, ignore_index=True)

    # Drop warm-up rows (EMA200 needs ~200 rows)
    master = master.dropna(subset=FEATURE_COLS + [TARGET])
    master = master[master[TARGET].notna()]

    # Save so Phase 3 is fast on next run
    os.makedirs("data/processed", exist_ok=True)
    master.to_csv(features_path, index=False)
    print(f"\n✅  Computed & saved features → {features_path}  ({len(master):,} rows)")
    return master


# ── CELL 5: Load Data ────────────────────────────────────────
master = load_master()
master.sort_values(["Ticker","Date"], inplace=True)
master.reset_index(drop=True, inplace=True)

# Ensure Cap_Encoded exists
cap_map = {"large": 0, "mid": 1, "small": 2}
if "Cap_Encoded" not in master.columns:
    master["Cap_Encoded"] = master["Cap_Type"].map(cap_map).fillna(0).astype(int)

# Ensure Target is integer
master[TARGET] = master[TARGET].astype(int)

print(f"\nDataset shape  : {master.shape}")
print(f"Tickers        : {master['Ticker'].nunique()}")
print(f"Date range     : {master['Date'].min().date()} → {master['Date'].max().date()}")
print(f"Class balance  : {master[TARGET].value_counts(normalize=True).round(3).to_dict()}")
if "Cap_Type" in master.columns:
    print(f"Cap breakdown  :\n{master.groupby('Cap_Type')['Ticker'].nunique().to_string()}")

# Sanity check — all feature columns present
missing = [c for c in FEATURE_COLS if c not in master.columns]
if missing:
    raise ValueError(f"Still missing feature columns after load: {missing}")
print(f"\n✅  All {len(FEATURE_COLS)} feature columns verified.\n")

# ── CELL 6: Train / Test Split ───────────────────────────────
def time_split(df: pd.DataFrame, test_ratio: float = 0.2):
    """Chronological split — no data leakage."""
    dates      = df["Date"].sort_values().unique()
    split_idx  = int(len(dates) * (1 - test_ratio))
    split_date = dates[split_idx]
    train = df[df["Date"] <  split_date].copy()
    test  = df[df["Date"] >= split_date].copy()
    return train, test, split_date

train_df, test_df, split_date = time_split(master)
print(f"Split date : {pd.Timestamp(split_date).date()}")
print(f"Train      : {len(train_df):,} rows  ({train_df['Date'].min().date()} → {train_df['Date'].max().date()})")
print(f"Test       : {len(test_df):,}  rows  ({test_df['Date'].min().date()}  → {test_df['Date'].max().date()})")

# ── CELL 7: Hyperparameter Grid ──────────────────────────────
PARAM_GRID = {
    "xgb__n_estimators"     : [200, 400, 600],
    "xgb__max_depth"        : [4, 6, 8],
    "xgb__learning_rate"    : [0.01, 0.05, 0.1],
    "xgb__subsample"        : [0.7, 0.8, 0.9],
    "xgb__colsample_bytree" : [0.6, 0.7, 0.8],
    "xgb__min_child_weight" : [1, 3, 5],
    "xgb__reg_alpha"        : [0, 0.1, 0.5],
    "xgb__reg_lambda"       : [1, 1.5, 2],
    "xgb__gamma"            : [0, 0.1, 0.2],
}

def build_pipeline() -> Pipeline:
    return Pipeline([
        ("scaler", RobustScaler()),
        ("xgb", XGBClassifier(
            objective         = "binary:logistic",
            eval_metric       = "logloss",
            use_label_encoder = False,
            random_state      = 42,
            tree_method       = "hist",
            device            = "cpu",
            # ADD THIS LINE ↓
            scale_pos_weight  = 1.6,
        ))
    ])

# ── CELL 8: Training Function ────────────────────────────────
def train_model(train: pd.DataFrame, name: str, n_iter: int = 25) -> Pipeline:
    X = train[FEATURE_COLS].values.astype(float)
    y = train[TARGET].values.astype(int)

    # Drop any remaining NaNs in X
    mask = ~np.isnan(X).any(axis=1)
    X, y = X[mask], y[mask]

    tscv   = TimeSeriesSplit(n_splits=5)
    search = RandomizedSearchCV(
        build_pipeline(), PARAM_GRID,
        n_iter=n_iter, cv=tscv, scoring="roc_auc",
        n_jobs=-1, verbose=0, random_state=42,
    )
    print(f"\n  [{name}]  Training on {len(X):,} samples ({n_iter} iterations)...")
    search.fit(X, y)
    print(f"  [{name}]  Best CV AUC : {search.best_score_:.4f}")
    return search.best_estimator_

# ── CELL 9: Evaluation Function ──────────────────────────────
def evaluate_model(model: Pipeline, test: pd.DataFrame, name: str) -> dict:
    X = test[FEATURE_COLS].values.astype(float)
    y = test[TARGET].values.astype(int)
    mask = ~np.isnan(X).any(axis=1)
    X, y = X[mask], y[mask]

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    m = {
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y, y_pred),  4),
        "Precision": round(precision_score(y, y_pred, zero_division=0), 4),
        "Recall"   : round(recall_score(y, y_pred,    zero_division=0), 4),
        "F1"       : round(f1_score(y, y_pred,         zero_division=0), 4),
        "ROC_AUC"  : round(roc_auc_score(y, y_prob),   4),
        "Test_Rows": int(len(y)),
    }

    print(f"\n{'='*55}")
    print(f"  {name}  —  TEST RESULTS")
    print(f"{'='*55}")
    for k, v in m.items():
        print(f"  {k:<12}: {v}")
    print(f"\n{classification_report(y, y_pred, target_names=['Bearish','Bullish'])}")

    # Confusion matrix
    cm  = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Bearish","Bullish"],
                yticklabels=["Bearish","Bullish"], ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    safe = name.lower().replace(" ","_")
    plt.savefig(f"plots/cm_{safe}.png", dpi=150)
    plt.close()
    return m

# ── CELL 10: SHAP Plot ───────────────────────────────────────
def plot_shap(model: Pipeline, test: pd.DataFrame, name: str):
    try:
        X_raw    = test[FEATURE_COLS].values.astype(float)[:500]
        X_scaled = model.named_steps["scaler"].transform(X_raw)
        explainer = shap.TreeExplainer(model.named_steps["xgb"])
        shap_vals = explainer.shap_values(X_scaled)
        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_vals, X_scaled,
                          feature_names=FEATURE_COLS, show=False)
        plt.title(f"SHAP Feature Importance — {name}")
        plt.tight_layout()
        plt.savefig(f"plots/shap_{name.lower().replace(' ','_')}.png", dpi=150)
        plt.close()
        print(f"  SHAP plot saved → plots/shap_{name.lower().replace(' ','_')}.png")
    except Exception as e:
        print(f"  SHAP skipped ({e})")

# ── CELL 11: Per-Cap Models ──────────────────────────────────
models_dict = {}
all_metrics = []
cap_splits  = {}

for cap in ["large", "mid", "small"]:
    cap_train = train_df[train_df["Cap_Type"] == cap]
    cap_test  = test_df[test_df["Cap_Type"]   == cap]

    if len(cap_train) < 500:
        print(f"\n⚠️  {cap} cap has only {len(cap_train)} training rows — skipping.")
        continue

    print(f"\n{'━'*55}")
    print(f"  TRAINING: {cap.upper()} CAP MODEL")
    print(f"{'━'*55}")

    model = train_model(cap_train, f"{cap.title()} Cap")
    m     = evaluate_model(model, cap_test, f"{cap.title()} Cap")
    all_metrics.append(m)

    path = f"models/xgb_{cap}.pkl"
    joblib.dump(model, path)
    print(f"  💾  Saved → {path}")

    models_dict[cap] = model
    cap_splits[cap]  = (cap_train, cap_test)

# ── CELL 12: Universal Model ─────────────────────────────────
print(f"\n{'━'*55}")
print(f"  TRAINING: UNIVERSAL MULTI-CAP MODEL")
print(f"{'━'*55}")
uni_model = train_model(train_df, "Universal", n_iter=30)
m_uni     = evaluate_model(uni_model, test_df, "Universal")
all_metrics.append(m_uni)
plot_shap(uni_model, test_df, "Universal")
joblib.dump(uni_model, "models/universal_xgb.pkl")
models_dict["universal"] = uni_model
print("  💾  Saved → models/universal_xgb.pkl")

# ── CELL 13: Save Report ─────────────────────────────────────
report_df = pd.DataFrame(all_metrics)
report_df.to_csv("reports/model_report.csv", index=False)
print(f"\n{'='*55}")
print("  ALL MODELS — COMPARISON")
print("="*55)
print(report_df.to_string(index=False))

# ── CELL 14: ROC Curve ───────────────────────────────────────
from sklearn.metrics import roc_curve

colors = {"large":"#2196F3","mid":"#4CAF50","small":"#FF9800","universal":"#E91E63"}
fig, ax = plt.subplots(figsize=(8, 6))

for cap, (_, cap_test) in cap_splits.items():
    model  = models_dict[cap]
    X      = cap_test[FEATURE_COLS].values.astype(float)
    y      = cap_test[TARGET].values.astype(int)
    mask   = ~np.isnan(X).any(axis=1)
    y_prob = model.predict_proba(X[mask])[:, 1]
    fpr, tpr, _ = roc_curve(y[mask], y_prob)
    auc = roc_auc_score(y[mask], y_prob)
    ax.plot(fpr, tpr, label=f"{cap.title()} Cap (AUC={auc:.3f})", color=colors[cap], lw=2)

X_u    = test_df[FEATURE_COLS].values.astype(float)
y_u    = test_df[TARGET].values.astype(int)
mask_u = ~np.isnan(X_u).any(axis=1)
y_prob_u = uni_model.predict_proba(X_u[mask_u])[:, 1]
fpr, tpr, _ = roc_curve(y_u[mask_u], y_prob_u)
ax.plot(fpr, tpr, label=f"Universal (AUC={roc_auc_score(y_u[mask_u],y_prob_u):.3f})",
        color=colors["universal"], lw=2.5, linestyle="--")

ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models"); ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("plots/roc_comparison.png", dpi=150)
plt.close()
print("\n  ROC plot saved → plots/roc_comparison.png")

# ── CELL 15: Per-Ticker Accuracy ─────────────────────────────
print("\nComputing per-ticker accuracy (universal model)...")
ticker_rows = []
for ticker, grp in tqdm(test_df.groupby("Ticker"), desc="Per-ticker eval"):
    if len(grp) < 10:
        continue
    X = grp[FEATURE_COLS].values.astype(float)
    y = grp[TARGET].values.astype(int)
    mask = ~np.isnan(X).any(axis=1)
    if mask.sum() < 5:
        continue
    yp = uni_model.predict(X[mask])
    ticker_rows.append({
        "Ticker"     : ticker,
        "Cap_Type"   : grp["Cap_Type"].iloc[0],
        "Accuracy"   : round(accuracy_score(y[mask], yp), 4),
        "Samples"    : int(mask.sum()),
        "Bullish_Pct": round(y[mask].mean(), 4),
    })

ticker_df = pd.DataFrame(ticker_rows).sort_values("Accuracy", ascending=False)
ticker_df.to_csv("reports/per_ticker_accuracy.csv", index=False)
print(f"\n  Top 10 by accuracy:")
print(ticker_df.head(10).to_string(index=False))
print(f"\n  Bottom 10 by accuracy:")
print(ticker_df.tail(10).to_string(index=False))

# ── DONE ─────────────────────────────────────────────────────
print("\n" + "="*55)
print("  PHASE 3 COMPLETE ✅")
print("="*55)
print("  Saved:")
print("    models/xgb_large.pkl  (if large-cap data available)")
print("    models/xgb_mid.pkl    (if mid-cap data available)")
print("    models/xgb_small.pkl  (if small-cap data available)")
print("    models/universal_xgb.pkl")
print("    reports/model_report.csv")
print("    reports/per_ticker_accuracy.csv")
print("    plots/  (confusion matrices, SHAP, ROC curves)")
print("\n➡  Next: Run phase4_finbert_sentiment.py")

✅  Loaded features file: 130,883 rows
    All 19 feature columns + Target present.

Dataset shape  : (130883, 29)
Tickers        : 134
Date range     : 2021-01-28 → 2024-12-31
Class balance  : {0: 0.616, 1: 0.384}
Cap breakdown  :
Cap_Type
large    49
mid      46
small    39

✅  All 19 feature columns verified.

Split date : 2024-03-20
Train      : 103,413 rows  (2021-01-28 → 2024-03-19)
Test       : 27,470  rows  (2024-03-20  → 2024-12-31)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TRAINING: LARGE CAP MODEL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [Large Cap]  Training on 40,131 samples (25 iterations)...
  [Large Cap]  Best CV AUC : 0.5287

  Large Cap  —  TEST RESULTS
  Model       : Large Cap
  Accuracy    : 0.5516
  Precision   : 0.3933
  Recall      : 0.4104
  F1          : 0.4017
  ROC_AUC     : 0.5265
  Test_Rows   : 10045

              precision    recall  f1-score   support

     Bearish       0.65      0.63      0.64      6361
     Bullish 